# Assembly
**Megahit**
https://www.metagenomics.wiki/tools/assembly/megahit
- de novo assembly (w/o reference genome)
- aligns/assembles short reads together to reconstruct one 'metagenome'
- assembled contigs are stored in fasta file

In [22]:
# Using trimmed, qc seqs from /trimmed
# separate into groups based on metadata 
    # spp x health status x sampledata - created in reads_counts. groups found in reads_meta
# 1)remove host from sample reads
# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)
# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, 
    #and ensures there are no gaps - larger portions of genomes if not all are now together in one sequence)
# 4)remove ITS2 seqs from assembled contigs (& remove adapters) 
    #... should try to perform on raw reads so we end up with just one final contig file 


In [23]:
# based on reads_meta groups, make folders and separate samples out

## Metadata and File Setup

In [36]:
import pandas as pd
import numpy as np
import os 
from pathlib import Path

In [25]:
os.chdir("/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw")

In [31]:
reads_meta = pd.read_csv("reads_meta.csv")
reads_meta.head()

,sampleid,raw,trimmed,pct_yield,Month_year,CollectionDate,Transect,TransectNum,NewTagNum,Species,SampleNum,Health_status,colony_id,group
0,052022_BEL_CBC_T1_10_PSTR,68572842,68449647,99.82,52022.0,5/21/22,CBC30N,1.0,4,PSTR,10.0,Diseased_Margin,T1_4_PSTR,52022_PSTR_Diseased_Margin
1,052022_BEL_CBC_T1_10_PSTR,68572842,68449647,99.82,52022.0,5/21/22,CBC30N,1.0,12,PSTR,10.0,Healthy,T1_12_PSTR,52022_PSTR_Healthy
2,052022_BEL_CBC_T1_11_PSTR,48741905,48660782,99.83,52022.0,5/21/22,CBC30N,1.0,4,PSTR,11.0,Diseased_Tissue,T1_4_PSTR,52022_PSTR_Diseased_Tissue
3,052022_BEL_CBC_T1_11_PSTR,48741905,48660782,99.83,52022.0,5/21/22,CBC30N,1.0,12,PSTR,11.0,Healthy,T1_12_PSTR,52022_PSTR_Healthy
4,052022_BEL_CBC_T1_12_MCAV,166267321,165624856,99.61,52022.0,5/21/22,CBC30N,1.0,8,MCAV,12.0,Diseased_Margin,T1_8_MCAV,52022_MCAV_Diseased_Margin


In [35]:
spp_list=reads_meta['Species'].unique()
print(spp_list)

['PSTR' 'MCAV' 'PAST' 'ORBI' 'MMEA' 'NEG']


In [33]:
reads_meta['group'].unique()

array(['52022_PSTR_Diseased_Margin', '52022_PSTR_Healthy',
       '52022_PSTR_Diseased_Tissue', '52022_MCAV_Diseased_Margin',
       '52022_MCAV_Diseased_Tissue', '52022_PAST_Healthy',
       '52022_OANN_Healthy', '52022_PAST_Diseased_Tissue',
       '52022_MCAV_Healthy', '52022_OFAV_Healthy',
       '52022_PAST_Diseased_Margin', '62019_MMEA_Healthy',
       '62019_PAST_Healthy', '62019_MCAV_Healthy', '102019_PSTR_Healthy',
       '122022_OANN_Diseased_Margin', '122022_PSTR_Healthy',
       '122022_OANN_Healthy', '122022_PSTR_Diseased_Tissue',
       '122022_PSTR_Diseased_Margin', '122022_PAST_Diseased_Margin',
       '122022_OANN_Diseased_Tissue', '122022_PAST_Diseased_Tissue',
       '122022_MCAV_Diseased_Tissue', '122022_PAST_Healthy',
       '122022_MCAV_Healthy', '122022_OFAV_Diseased_Margin',
       '122022_OFAV_Diseased_Tissue', '122022_OFAV_Healthy',
       '122022_MCAV_Diseased_Margin', 'Negative'], dtype=object)

In [ ]:
# make sample list for each spp and group? 

In [ ]:
# common variables to use in scripts
BASE_DIR = "/scratch4/workspace/brooke_sienkiewicz_student_uml_edu-bel_raw"
STOREREADS = "reads_filtered.txt" # read_count,step,sampleid

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu-long  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/pilot_past/slurm-assembly-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8

# 1)remove host from sample reads
# Host seq removal - Thij's script https://github.com/ThijsSt/SCTLD-metagenomes/blob/main/Quality_control_metagenomes.ipynb

#set general parameters:
SAMPLENAME="pp"
GENOME="pp"
READSPATH='/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/pilot_past/trimmed_redo'
INDEX="$GENOME"_DB
INPUTPATH="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/Mcav_genome"
OUTDIR='/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/pilot_past/assembly'
WORKINGPATH='/project/pi_sarah_gignouxwolfsohn_uml_edu/brooke/pilot_past/assembly/host_removed'
# testing script 

# gonna need to loop through each spp 

# get unique species from list of samples, spp, and groups
spp_list=$(cut -f 2 filtered_sample_groups.txt | tail -n +2 | sort -u)

# loop through spp list...
for spp in $spp_list; do
    # make spp folder if it doesn't already exist 
    mkdir -p "$spp"

    # identify samples for each spp to host remove together 
    samples=$(awk -F'\t' -v s="$spp" '$2 == s {print $1}' filtered_sample_groups.txt)
    expected_count=$(( $(echo $samples | wc -w) * 2)) # 69 for mcav (double for f & r)

    # move sampleids for each spp into separate folders
    for id in $samples; do
        # Create a symbolic link (shortcut) of the seq files into the species folder
        ln -s ../trimmed/${id}_R1_001_val_1.fq ./${spp}/${id}_R1_001_val_1.fq
        ln -s ../trimmed/${id}_R2_001_val_1.fq ./${spp}/${id}_R2_001_val_1.fq
    done

    # check correct # of samples was moved
    actual_samples=$(ls "$spp" | wc -l)
    if [("$expected_count" == "$actual_samples")]; then
        echo "SUCCESS: all $expected_count samples found"
    else 
        echo "ERROR: Expected $expected_count samples, but only found $actual_samples





done






# old - have not updated 
mkdir $WORKINGPATH

#build a bowtie2 index from a known genome
bowtie2-build $INPUTPATH/Mcavernosa_July2018.fasta $INPUTPATH/"$INDEX"

#loop through samples
while IFS= read -r SAMPLEID; do

#re-align reads back to the index
bowtie2 -p 8 -x $INPUTPATH/$INDEX -1 "$READSPATH"/"${SAMPLEID}_R1_001_val_1.fq" -2 "$READSPATH"/"${SAMPLEID}_R2_001_val_2.fq" -S $WORKINGPATH/"${SAMPLEID}"_mapped_and_unmapped.sam

#convert sam file from bowtie to a bam file for processing
samtools view -bS $WORKINGPATH/"${SAMPLEID}"_mapped_and_unmapped.sam > $WORKINGPATH/"${SAMPLEID}"_mapped_and_unmapped.bam

#extract only the reads of which both do not match against the host genome
samtools view -b -f 12 -F 256 $WORKINGPATH/"${SAMPLEID}"_mapped_and_unmapped.bam > $WORKINGPATH/"${SAMPLEID}"_bothReadsUnmapped.bam
#ask thijs what flags mean 

# sorts the file so both mates are together and then extracts them back as .fastq files
samtools sort -n -m 5G -@ 2 $WORKINGPATH/"${SAMPLEID}"_bothReadsUnmapped.bam -o $WORKINGPATH/"${SAMPLEID}"_bothReadsUnmapped_sorted.bam
samtools fastq -@ 8 $WORKINGPATH/"${SAMPLEID}"_bothReadsUnmapped_sorted.bam \
    -1 $WORKINGPATH/"${SAMPLEID}"_host_removed_R1.fastq \
    -2 $WORKINGPATH/"${SAMPLEID}"_host_removed_R2.fastq \
    -0 /dev/null -s /dev/null -n
#can i direct these to a diff folder?

done < "pp_sampleids.txt"
#run in dir with sampleids txt file 

conda deactivate
conda activate assembly

# 2)concatenate all f and r seqs into single file (1 for f, 1 for r)

# Read the sample IDs from the file
while IFS= read -r SAMPLEID; do
    # Construct the file paths for forward and reverse reads
    FORWARD_READ="$WORKINGPATH/${SAMPLEID}_host_removed_R1.fastq"
    REVERSE_READ="$WORKINGPATH/${SAMPLEID}_host_removed_R2.fastq"

    # Check if the files exist before concatenating
    if [ -e "$FORWARD_READ" ]; then
        cat "$FORWARD_READ" >> "$OUTDIR/${SAMPLENAME}_reads_R1_ALL.fastq"
    else
        echo "Forward read file not found for sample $SAMPLEID"
    fi

    if [ -e "$REVERSE_READ" ]; then
        cat "$REVERSE_READ" >> "$OUTDIR/${SAMPLENAME}_reads_R2_ALL.fastq"
    else
        echo "Reverse read file not found for sample $SAMPLEID"
    fi
done < "pp_sampleids.txt"

# 3)ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
megahit --presets meta-large \
-1 "$OUTDIR"/"$SAMPLENAME"_reads_R1_ALL.fastq \
-2 "$OUTDIR"/"$SAMPLENAME"_reads_R2_ALL.fastq \
--keep-tmp-files \
-o megahit_host_removed --out-prefix $SAMPLENAME \
--continue
#this one has to make the directory; will fail if it already exists

# try metavelvet next? 



# JOB-ID:
# bash script file name: 